# Figures for Lesson 1.1 - Introduction to Statistics

This notebook draws the two figures used in `../1-1-introduction_to_statistics.md`:

- `skewness.png` / `.svg`: left-skewed, symmetric and right-skewed curves with the mean, median and mode, and the matching box plots.
- `box_plot_outliers.png` / `.svg`: box plots for the four cases (no outliers, max outlier, min outlier, both).
- `histogram_vs_bar_graph.png` / `.svg`: the test-score histogram next to a bar graph of the same counts.

Open it from this `figures` folder (Jupyter does that by default) and run all cells. The images are saved next to the notebook.

Requirements: `numpy`, `scipy`, `matplotlib`.

In [ ]:
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
from scipy import stats

OUT = Path.cwd()  # the folder this notebook is in
plt.rcParams.update({"font.size": 11, "axes.spines.top": False, "axes.spines.right": False})
COLOR = {"mean": "#D55E00", "median": "#0072B2", "mode": "#009E73"}
STYLE = {"mean": "--", "median": "-", "mode": ":"}


def save(fig, name):
    """Save a figure as PNG and SVG next to this notebook, then show it."""
    fig.savefig(OUT / f"{name}.png", dpi=200, bbox_inches="tight", facecolor="white")
    fig.savefig(OUT / f"{name}.svg", bbox_inches="tight", facecolor="white")
    plt.show()

## Skewness

Right skew uses a Gamma(2, 1) distribution (mode 1, mean 2). Left skew is its mirror image. Symmetric is the standard normal curve. Whiskers are drawn to the 1.5 x IQR fences.

In [ ]:
g = stats.gamma(a=2)
g_q1, g_med, g_q3 = g.ppf([0.25, 0.5, 0.75])
W = 7  # gamma panels are drawn on [0, 7]

panels = [
    dict(title="Skewed left (negative skew)", x=np.linspace(0, W, 400), pdf=lambda x: g.pdf(W - x),
         mode=W - 1, median=W - g_med, mean=W - 2, q1=W - g_q3, q3=W - g_q1,
         order="mean < median < mode"),
    dict(title="Symmetric", x=np.linspace(-3.5, 3.5, 400), pdf=stats.norm.pdf,
         mode=0, median=0, mean=0, q1=stats.norm.ppf(0.25), q3=stats.norm.ppf(0.75),
         order="mean = median = mode"),
    dict(title="Skewed right (positive skew)", x=np.linspace(0, W, 400), pdf=g.pdf,
         mode=1, median=g_med, mean=2, q1=g_q1, q3=g_q3,
         order="mode < median < mean"),
]

fig, axes = plt.subplots(2, 3, figsize=(13, 5.8), sharex="col",
                         gridspec_kw={"height_ratios": [3, 1]})
for col, p in enumerate(panels):
    ax, bx = axes[0, col], axes[1, col]
    y = p["pdf"](p["x"])
    ax.fill_between(p["x"], y, color="#E8EEF4")
    ax.plot(p["x"], y, color="#333333", lw=2)
    for k in ("mode", "median", "mean"):
        ax.axvline(p[k], color=COLOR[k], ls=STYLE[k], lw=2.2, label=k.capitalize())
    ax.set(title=p["title"], yticks=[], ylim=(0, y.max() * 1.15))
    ax.spines["left"].set_visible(False)
    ax.set_xlabel(p["order"], fontweight="bold")

    iqr = p["q3"] - p["q1"]
    lo = max(p["x"][0], p["q1"] - 1.5 * iqr)
    hi = min(p["x"][-1], p["q3"] + 1.5 * iqr)
    bx.bxp([dict(med=p["median"], q1=p["q1"], q3=p["q3"], whislo=lo, whishi=hi, fliers=[])],
           vert=False, widths=0.6, patch_artist=True, showfliers=False,
           boxprops=dict(facecolor="#BFD3E6"), medianprops=dict(color=COLOR["median"], lw=2.5))
    bx.set(yticks=[])
    bx.spines["left"].set_visible(False)

axes[0, 0].legend(loc="upper left", frameon=False)
fig.suptitle("Skewness: where the mean, median and mode sit, and what the box plot looks like",
             fontweight="bold")
fig.tight_layout()
save(fig, "skewness")

## Box plots: is the min or the max an outlier?

The data are the worked examples from the lesson. Quartiles use the lesson's method (medians of the lower and upper halves, leaving out the overall median when n is odd).

In [ ]:
cases = [
    ("(a) No outliers: whiskers reach the min and the max",
     [5, 7, 8, 10, 11, 13, 14, 16, 16, 17, 27], (8, 13, 16)),
    ("(b) Max is an outlier: right whisker stops at the largest non-outlier",
     [13, 16, 18, 18, 22, 23, 25, 28, 29, 31, 38, 50], (18, 24, 30)),
    ("(c) Min is an outlier: left whisker stops at the smallest non-outlier",
     [2, 15, 17, 18, 19, 20, 21, 22, 23, 25, 26], (17, 20, 23)),
    ("(d) Min and max are both outliers: both whiskers stop short",
     [2, 15, 17, 18, 19, 20, 21, 22, 23, 25, 40], (17, 20, 23)),
]

fig, axes = plt.subplots(len(cases), 1, figsize=(11, 10.5))
for ax, (title, data, (q1, q2, q3)) in zip(axes, cases):
    data = np.array(data)
    iqr = q3 - q1
    lower_fence, upper_fence = q1 - 1.5 * iqr, q3 + 1.5 * iqr
    is_out = (data < lower_fence) | (data > upper_fence)
    inside, outliers = data[~is_out], data[is_out]

    ax.bxp([dict(med=q2, q1=q1, q3=q3, whislo=inside.min(), whishi=inside.max(), fliers=outliers)],
           positions=[0], vert=False, widths=0.5, patch_artist=True,
           boxprops=dict(facecolor="#BFD3E6"), medianprops=dict(color="#0072B2", lw=2.5),
           flierprops=dict(marker="o", markerfacecolor="none", markeredgecolor="#D55E00",
                           markersize=9, markeredgewidth=2))

    # the data as dots above the box (repeated values are stacked)
    values, counts = np.unique(data, return_counts=True)
    for v, c in zip(values, counts):
        dot_color = "#D55E00" if (v < lower_fence or v > upper_fence) else "#555555"
        ax.scatter([v] * c, 0.55 + 0.13 * np.arange(c), color=dot_color, s=25, zorder=3)

    for fence, name in ((lower_fence, "lower fence"), (upper_fence, "upper fence")):
        ax.axvline(fence, color="#888888", ls="--", lw=1.2)
        is_lower = name == "lower fence"
        ax.annotate(f"{name} = {fence:g}", (fence, 0.97), xycoords=("data", "axes fraction"),
                    xytext=(-4 if is_lower else 4, 0), textcoords="offset points",
                    ha="right" if is_lower else "left", va="top", color="#888888", fontsize=9)

    ax.set_title(title, loc="left", fontweight="bold")
    ax.set(yticks=[], ylim=(-0.6, 1.25))
    ax.spines["left"].set_visible(False)

fig.text(0.5, 0.0, "Dots = the data (outliers in orange). Dashed lines = fences Q1 - 1.5 x IQR and Q3 + 1.5 x IQR.",
         ha="center", color="#555555", fontsize=9)
fig.tight_layout()
save(fig, "box_plot_outliers")

## Histogram vs bar graph

The 15 test scores from the lesson, grouped into the classes 60-69, 70-79, 80-89 and 90-100. The histogram's bars touch because the classes are consecutive ranges on a number line; the bar graph of letter grades has gaps because the grades are separate categories.

In [ ]:
scores = np.array([65, 72, 93, 68, 76, 98, 84, 85, 79, 88, 90, 82, 83, 87, 78])
edges = [60, 70, 80, 90, 100]          # np.histogram puts 100 in the last class
freq, _ = np.histogram(scores, bins=edges)
assert list(freq) == [2, 4, 6, 3]

fig, (ax, bx) = plt.subplots(1, 2, figsize=(12, 4.8))

ax.hist(scores, bins=edges, color="#4C8BC4", edgecolor="#1F4E79", linewidth=1.5)
for left, f in zip(edges[:-1], freq):
    ax.text(left + 5, f + 0.12, str(f), ha="center", fontweight="bold")
ax.plot(scores, np.full(len(scores), -0.35), "|", color="#D55E00", markersize=12, mew=2, clip_on=False)
ax.set(title="Histogram\nnumeric classes, bars touch", xticks=edges, ylim=(0, 7), xlim=(60, 100),
       xlabel="Test score (orange ticks = the 15 individual scores)", ylabel="Frequency (number of students)")

grades = ["D\n(60-69)", "C\n(70-79)", "B\n(80-89)", "A\n(90-100)"]
bx.bar(grades, freq, width=0.6, color="#4C8BC4", edgecolor="#1F4E79", linewidth=1.5)
for i, f in enumerate(freq):
    bx.text(i, f + 0.12, str(f), ha="center", fontweight="bold")
bx.set(title="Bar graph\nseparate categories, gaps between bars", ylim=(0, 7),
       xlabel="Letter grade", ylabel="Frequency (number of students)")

for a in (ax, bx):
    a.yaxis.grid(True, color="#E3E3E3")
    a.set_axisbelow(True)
fig.suptitle("Test scores of 15 students: histogram vs bar graph", fontweight="bold")
fig.tight_layout()
save(fig, "histogram_vs_bar_graph")